<a href="https://colab.research.google.com/github/sidatKS/Anthropic_CPN/blob/main/Prompt_Eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
!pip install anthropic

In [12]:
from google.colab import userdata
import os
os.environ["ANTHROPIC_API_KEY"] = userdata.get('ANTHROPIC_API_KEY')
from anthropic import Anthropic
client = Anthropic()  # auto-picks up the env var
model = "claude-haiku-4-5"

In [13]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [14]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [15]:
dataset = generate_dataset()
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [16]:
def run_prompt(test_case):
  """Merges the prompt and tasks from dataset, then returns the result"""
  prompt = f"""
Please solve the following task:
{test_case['task']}
"""
  messages = []
  add_user_message(messages, prompt)
  add_assistant_message(messages, "```python")
  output = chat(messages, stop_sequences=["```"])
  return output, prompt

In [17]:
def run_test_case(test_case):
  """calls run_prompt, then grades the results"""
  output, prompt = run_prompt(test_case)
  #TODO GRADING
  score = 10
  return{
      "test_case": test_case,
      "prompt": prompt,
      "output": output,
      "score": score

  }

In [18]:
def run_eval(dataset):
  """Loads the dataset and calls run_test_case with each case"""
  results = []
  for test_case in dataset:
    results.append(run_test_case(test_case))
  return results


In [19]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)
results = run_eval(dataset)

In [20]:
print(json.dumps(results, indent=2))

[
  {
    "test_case": {
      "task": "Write a Python function that validates an AWS S3 bucket name according to AWS naming rules (3-63 characters, lowercase letters, numbers, and hyphens only, must start and end with letter or number)"
    },
    "prompt": "\nPlease solve the following task:\nWrite a Python function that validates an AWS S3 bucket name according to AWS naming rules (3-63 characters, lowercase letters, numbers, and hyphens only, must start and end with letter or number)\n",
    "output": "\nimport re\n\ndef validate_s3_bucket_name(bucket_name: str) -> tuple[bool, str]:\n    \"\"\"\n    Validates an AWS S3 bucket name according to AWS naming rules.\n    \n    AWS S3 Bucket Naming Rules:\n    - Must be between 3 and 63 characters long\n    - Can only contain lowercase letters, numbers, and hyphens (-)\n    - Must start with a lowercase letter or number\n    - Must end with a lowercase letter or number\n    - Cannot contain consecutive hyphens\n    - Cannot be formatted 

### Function Flow Map

```mermaid
graph TD
    A[Start Evaluation] --> B[generate_dataset()];
    B --> C[dataset.json];
    C --> D[Load dataset.json (in run_eval)];
    D --> E[run_eval(dataset)];
    E --> F{For each test_case in dataset};
    F --> G[run_test_case(test_case)];
    G --> H[run_prompt(test_case)];
    H --> I[Anthropic chat(messages, ...)];
    I --> J[Claude API (claude-haiku-4-5)];
    J --> K[Return output, prompt from Claude];
    K --> L[Return {output, test_case, prompt, score} from run_test_case];
    L --> M[Collect results in run_eval];
    M --> N[End Evaluation / Return results];
    N --> O[print(json.dumps(results))];

    style A fill:#f9f,stroke:#333,stroke-width:2px;
    style B fill:#bbf,stroke:#333,stroke-width:2px;
    style C fill:#ccf,stroke:#333,stroke-width:2px;
    style D fill:#bbf,stroke:#333,stroke-width:2px;
    style E fill:#bbf,stroke:#333,stroke-width:2px;
    style F fill:#ffa,stroke:#333,stroke-width:2px;
    style G fill:#bbf,stroke:#333,stroke-width:2px;
    style H fill:#bbf,stroke:#333,stroke-width:2px;
    style I fill:#bbf,stroke:#333,stroke-width:2px;
    style J fill:#fcc,stroke:#333,stroke-width:2px;
    style K fill:#ddf,stroke:#333,stroke-width:2px;
    style L fill:#ddf,stroke:#333,stroke-width:2px;
    style M fill:#bbf,stroke:#333,stroke-width:2px;
    style N fill:#f9f,stroke:#333,stroke-width:2px;
    style O fill:#ccf,stroke:#333,stroke-width:2px;
```